In [ ]:
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

sns.set_context("poster")
sns.set_style("ticks")

In [ ]:
fi = pd.read_parquet("0.parquet")
fi = fi[fi.split == "test"]
spearman = pd.read_parquet("1.parquet")
spearman = spearman[spearman.split == "test"]
weights = pd.read_parquet("2.parquet")
weights["AbsWeight"] = weights.Weight.abs()

In [ ]:
id_cols = [
    "trainer.model_builder.param",
    "trainer.representations.noise_level",
    "trainer.representations.seed",
]

In [ ]:
features_fi = (
    fi[fi["trainer.representations.noise_level"] == 0]
    .groupby(["Feature", "trainer.model_builder.param"])["mean"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(5)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features_weights = (
    weights[weights["trainer.representations.noise_level"] == 0.1]
    .groupby(["Feature", "trainer.model_builder.param"])
    .AbsWeight.mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(5)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features = pd.concat([features_fi, features_weights])[["Feature"]].drop_duplicates()

# FI

In [ ]:
g = sns.relplot(
    fi.merge(features),
    kind="line",
    x="trainer.representations.noise_level",
    y="mean",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_ylabels("FI")
g.set_xlabels("Artificial Noise Level")
g.set_titles("{col_name}")
g.refline(y=0, linestyle="--", color="black", linewidth=1)

# Weights

In [ ]:
g = sns.relplot(
    weights.merge(features),
    kind="line",
    x="trainer.representations.noise_level",
    y="Weight",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_xlabels("Artificial Noise Level")
g.set_titles("{col_name}")
g.refline(y=0, linestyle="--", color="black", linewidth=1)

# Training duration

In [ ]:
ax = sns.lineplot(
    weights,
    x="trainer.representations.noise_level",
    y="training_duration",
    hue="trainer.model_builder.param",
    errorbar="sd",
    marker="o",
)
ax.set_ylabel("Training Duration (s)")
ax.set_xlabel("Artificial Noise Level")
ax.get_legend().set_title(None)

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman,
    x="trainer.representations.noise_level",
    y="mean",
    hue="trainer.model_builder.param",
    errorbar="sd",
    marker="o",
)
ax.set_ylabel("Spearman")
ax.set_xlabel("Artificial Noise level")
ax.get_legend().set_title(None)

# Weights discrepancy

## Between MLEM and FR-RSA

In [ ]:
compare_weights = weights.pivot(
    index=[
        "Feature",
        "trainer.representations.seed",
        "trainer.representations.noise_level",
    ],
    columns=["trainer.model_builder.param"],
    values="Weight",
).reset_index()
compare_weights = (
    compare_weights.groupby(
        [
            "trainer.representations.seed",
            "trainer.representations.noise_level",
        ],
    )
    .apply(
        lambda x: pd.Series(
            {
                "Weighted $\\tau$": stats.weightedtau(
                    x.cholesky.abs(), x.triu.abs()
                ).statistic,
                "L2": ((x.cholesky - x.triu) ** 2).sum() ** 0.5,
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
compare_weights["Type"] = "Weights"

compare_fis = fi.pivot(
    index=[
        "Feature",
        "trainer.representations.seed",
        "trainer.representations.noise_level",
    ],
    columns=["trainer.model_builder.param"],
    values="mean",
).reset_index()
compare_fis = (
    compare_fis.groupby(
        ["trainer.representations.seed", "trainer.representations.noise_level"],
    )
    .apply(
        lambda x: pd.Series(
            {
                "Weighted $\\tau$": stats.weightedtau(x.cholesky, x.triu).statistic,
                "L2": ((x.cholesky - x.triu) ** 2).sum() ** 0.5,
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
compare_fis["Type"] = "FIs"

compare = pd.concat([compare_weights, compare_fis])

In [ ]:
ax = sns.lineplot(
    compare,
    x="trainer.representations.noise_level",
    y="Weighted $\\tau$",
    hue="Type",
    style="Type",
    markers=True,
    dashes=False,
    errorbar="sd",
)
sns.move_legend(ax, loc="best", title=None)
ax.set_xlabel("Noise Level")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/artificial_noise/weighted_tau_mlem_frrsa.pdf",
    bbox_inches="tight",
)
plt.show()

In [ ]:
ax = sns.lineplot(
    compare,
    x="trainer.representations.noise_level",
    y="L2",
    hue="Type",
    style="Type",
    markers=True,
    dashes=False,
    errorbar="sd",
)
sns.move_legend(ax, loc="best", title=None)
ax.set_xlabel("Noise Level")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/artificial_noise/l2_mlem_frrsa.pdf",
    bbox_inches="tight",
)
plt.show()

## w.r.t. FIs/weights without noise

In [ ]:
weights_no_noise = weights.loc[
    weights["trainer.representations.noise_level"] == 0,
    ["Feature", "Weight"] + id_cols,
]
weights_no_noise = weights_no_noise.drop(columns=["trainer.representations.noise_level"])
weights_no_noise = weights_no_noise.rename(columns={"Weight": "Weight_no_noise"})
compare_weights = weights.merge(weights_no_noise)
compare_weights = (
    compare_weights.groupby(
        id_cols,
    )
    .apply(
        lambda x: pd.Series(
            {
                "Weighted $\\tau$": stats.weightedtau(
                    x.Weight_no_noise.abs(), x.Weight.abs()
                ).statistic,
                "L2": ((x.Weight_no_noise - x.Weight) ** 2).sum() ** 0.5,
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
compare_weights["Type"] = "Weights"

fi_no_noise = fi.loc[
    fi["trainer.representations.noise_level"] == 0,
    ["Feature", "mean"] + id_cols,
]
fi_no_noise = fi_no_noise.drop(columns=["trainer.representations.noise_level"])
fi_no_noise = fi_no_noise.rename(columns={"mean": "mean_no_noise"})
compare_fis = fi.merge(fi_no_noise)
compare_fis = (
    compare_fis.groupby(
        id_cols,
    )
    .apply(
        lambda x: pd.Series(
            {
                "Weighted $\\tau$": stats.weightedtau(
                    x.mean_no_noise, x["mean"]
                ).statistic,
                "L2": ((x.mean_no_noise - x["mean"]) ** 2).sum() ** 0.5,
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
compare_fis["Type"] = "FIs"

compare = pd.concat([compare_weights, compare_fis], axis=0)

In [ ]:
ax = sns.lineplot(
    compare,
    x="trainer.representations.noise_level",
    y="Weighted $\\tau$",
    hue="trainer.model_builder.param",
    style="Type",
    marker="o",
    errorbar="sd",
)
ax.get_legend().set_title(None)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_xlabel("Noise level")
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/artificial_noise/weighted_tau.pdf",
    bbox_inches="tight",
)
plt.show()

In [ ]:
ax = sns.lineplot(
    compare,
    x="trainer.representations.noise_level",
    y="L2",
    hue="trainer.model_builder.param",
    style="Type",
    marker="o",
    errorbar="sd",
)
ax.get_legend().set_title(None)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_xlabel("Noise level")
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/artificial_noise/l2.pdf",
    bbox_inches="tight",
)
plt.show()